In [1]:

# Cell 1 — Setup, imports, Google Drive

from __future__ import annotations

import os
import gc
import json
import math
import time
import random
import hashlib
import shutil
import zipfile
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any, Iterable

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
)
from sklearn.preprocessing import MinMaxScaler

try:
    from IPython.display import display
except Exception:
    display = print

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Không chạy trong Google Colab; bỏ qua mount Drive.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {props.total_memory / 1e9:.1f} GB")


Mounted at /content/drive
Device: cuda | PyTorch: 2.11.0+cu128
GPU: NVIDIA A100-SXM4-40GB | VRAM: 42.4 GB


In [2]:

# Cell 2 — Master configuration

RUN_MODE = "all"  # audit | batch_partition | relation_ablation | aggregate | all
VALID_RUN_MODES = {"audit", "batch_partition", "relation_ablation", "aggregate", "all"}
if RUN_MODE not in VALID_RUN_MODES:
    raise ValueError(f"RUN_MODE must be one of {sorted(VALID_RUN_MODES)}")

DATASETS_TO_RUN = ["CIC_V2", "UNSW_V2"]
MODEL_SEEDS = [42, 123, 456]
BENCHMARK_SPLIT_SEED = 2027
PROTOCOL_NAME = "exact_feature_group"

DRIVE_RUN_ROOT = Path("/content/drive/MyDrive/THAT_IDS_RERUN")
STEP4_ROOT = DRIVE_RUN_ROOT / "conference_step4_exact_feature_group_v1"
STEP4_SHARED_CACHE_ROOT = DRIVE_RUN_ROOT / "conference_step4_shared_feature_cache_v1"
STEP5_ROOT = DRIVE_RUN_ROOT / "conference_step5_literature_subset_v1"
STEP5_CONFIRMATION_ROOT = STEP5_ROOT / "confirmation"
STEP5_SUBSET_DEFINITION_ROOT = STEP5_ROOT / "subset_definition"
STEP5_SUBSET_CACHE_ROOT = STEP5_ROOT / "subset_feature_cache"
STEP6_ROOT = DRIVE_RUN_ROOT / "conference_step6_public_baselines_v1"
STEP7_ROOT = DRIVE_RUN_ROOT / "conference_step7_tabular_loo_v1"

STEP8_EXPERIMENT_TAG = "conference_step8_supervisor_revision_master_v1"
OUTPUT_ROOT = DRIVE_RUN_ROOT / STEP8_EXPERIMENT_TAG
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

DATA_PATH_CANDIDATES = {
    "UNSW_V2": [
        "/content/drive/MyDrive/datasets/NF-UNSW-NB15-v2.csv",
        "/content/drive/MyDrive/NF-UNSW-NB15-v2.csv",
    ],
    "CIC_V2": [
        "/content/drive/MyDrive/datasets/NF-CSE-CIC-IDS2018-v2.csv",
        "/content/drive/MyDrive/NF-CSE-CIC-IDS2018-v2.csv",
    ],
}

# New experiments
BATCH_PARTITION_SEEDS = [0, 3027, 4027]  # 0 = exact original Step-5 order, reused
RELATION_VARIANTS_TO_RUN = [
    "ip_ip_only",
    "ip_protocol_only",
    "both_no_protocol_numeric",
]
MAX_EPOCHS = 80
PATIENCE = 12
SAVE_LAST_EVERY = 1
WINDOW_SIZE = 5000

# Model constants: exact Step-5 architecture
HIDDEN_DIM = 96
AE_HIDDEN = 192
LATENT_DIM = 96
GRAPH_DROPOUT = 0.10
ENC_DROPOUT = 0.10
CLS_DROPOUT = 0.15
INIT_GATE_LOGIT = -1.5
CTX_RECON_WEIGHT = 0.10

LEARNING_RATE = 5e-5
WEIGHT_DECAY = 1e-5
GRAD_CLIP_NORM = 1.0
LAMBDA_RECON = 1.0
LAMBDA_CLS = 1.0
FOCAL_ALPHA = 0.75
FOCAL_GAMMA = 2.0
POS_WEIGHT_CAP = 12.0
SCORE_CLS_WEIGHT = 0.90
SCORE_RECON_WEIGHT = 0.10
CLIP_MIN, CLIP_MAX = -10.0, 10.0

# Audit / post-processing
CSV_CHUNK_SIZE = 500_000
BETA_VALUES = [0.0, 0.25, 0.50, 0.75, 0.90, 1.0]
FPR_TARGETS = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2]
RECALL_TARGETS = [0.90, 0.95, 0.99]
TEST_CURVE_FPR_TARGETS = FPR_TARGETS
TEST_CURVE_RECALL_TARGETS = RECALL_TARGETS
BOOTSTRAP_REPLICATES = 1000
BOOTSTRAP_SEED = 2027
RUN_GROUP_BOOTSTRAP = True
HASH_VERIFY_MODE = "sample"  # none | sample | full
HASH_VERIFY_SAMPLE_GROUPS = 10000
STRICT_PREFLIGHT = True

# Optional fast debug
SMOKE_MODE = False
SMOKE_MAX_ROWS_PER_SPLIT = 20000
if SMOKE_MODE:
    MAX_EPOCHS = 2
    PATIENCE = 2
    MODEL_SEEDS = [42]
    BATCH_PARTITION_SEEDS = [0, 3027]
    BOOTSTRAP_REPLICATES = 50

print("Step 8 output:", OUTPUT_ROOT)
print("RUN_MODE:", RUN_MODE)
print("Datasets:", DATASETS_TO_RUN)
print("Model seeds:", MODEL_SEEDS)
print("Batch partitions:", BATCH_PARTITION_SEEDS)
print("Relation variants:", RELATION_VARIANTS_TO_RUN)


Step 8 output: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step8_supervisor_revision_master_v1
RUN_MODE: all
Datasets: ['CIC_V2', 'UNSW_V2']
Model seeds: [42, 123, 456]
Batch partitions: [0, 3027, 4027]
Relation variants: ['ip_ip_only', 'ip_protocol_only', 'both_no_protocol_numeric']


In [3]:

# Cell 3 — Paths, fingerprints, atomic files, RNG save/restore

def resolve_dataset_path(dataset_name: str) -> str:
    for candidate in DATA_PATH_CANDIDATES.get(dataset_name, []):
        if Path(candidate).exists():
            return candidate
    raise FileNotFoundError(
        "Không tìm thấy dataset " + dataset_name + ". Đã thử:\n"
        + "\n".join(DATA_PATH_CANDIDATES.get(dataset_name, []))
    )

def step5_dataset_dir(dataset_name: str) -> Path:
    return STEP5_CONFIRMATION_ROOT / dataset_name

def step5_shared_dir(dataset_name: str) -> Path:
    return (
        step5_dataset_dir(dataset_name)
        / PROTOCOL_NAME / "shared" / f"seed_{BENCHMARK_SPLIT_SEED}"
    )

def subset_definition_dir(dataset_name: str) -> Path:
    return STEP5_SUBSET_DEFINITION_ROOT / dataset_name

def step5_full_task_dir(dataset_name: str, seed: int) -> Path:
    return step5_dataset_dir(dataset_name) / PROTOCOL_NAME / "full" / f"seed_{seed}"

def step7_task_dir(dataset_name: str, model_name: str, seed: int) -> Path:
    return STEP7_ROOT / dataset_name / PROTOCOL_NAME / model_name / f"seed_{seed}"

def batch_task_dir(dataset_name: str, partition_seed: int, model_seed: int) -> Path:
    return (
        OUTPUT_ROOT / "batch_partition_sensitivity" / dataset_name
        / f"partition_{partition_seed}" / "full" / f"seed_{model_seed}"
    )

def relation_task_dir(dataset_name: str, variant_name: str, model_seed: int) -> Path:
    return (
        OUTPUT_ROOT / "relation_ablation" / dataset_name
        / variant_name / f"seed_{model_seed}"
    )

def load_json(path: Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def atomic_json_dump(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)

def atomic_torch_save(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def load_torch_checkpoint(path: Path, map_location: torch.device) -> Dict[str, Any]:
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def config_fingerprint(payload: Dict[str, Any]) -> str:
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True, default=str).encode("utf-8")
    ).hexdigest()[:24]

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def capture_rng_state() -> Dict[str, Any]:
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state

def _cpu_byte_tensor(value: Any) -> torch.Tensor:
    if isinstance(value, torch.Tensor):
        return value.detach().to(device="cpu", dtype=torch.uint8).contiguous()
    return torch.as_tensor(value, dtype=torch.uint8, device="cpu").contiguous()

def restore_rng_state(state: Optional[Dict[str, Any]]) -> None:
    if not state:
        return
    if "python" in state:
        random.setstate(state["python"])
    if "numpy" in state:
        np.random.set_state(state["numpy"])
    if "torch" in state:
        torch.set_rng_state(_cpu_byte_tensor(state["torch"]))
    if torch.cuda.is_available() and "cuda" in state:
        values = state["cuda"]
        if isinstance(values, torch.Tensor):
            values = [values]
        for device_index, cuda_state in enumerate(list(values)[:torch.cuda.device_count()]):
            torch.cuda.set_rng_state(_cpu_byte_tensor(cuda_state), device=device_index)

def update_master_progress(phase: str, dataset: Optional[str] = None, status: str = "completed",
                           details: Optional[Dict[str, Any]] = None) -> None:
    path = OUTPUT_ROOT / "master_progress.json"
    progress = load_json(path) if path.exists() else {
        "experiment_tag": STEP8_EXPERIMENT_TAG,
        "created_at_unix": time.time(),
        "phases": {},
    }
    key = phase if dataset is None else f"{phase}:{dataset}"
    progress["phases"][key] = {
        "status": status,
        "updated_at_unix": time.time(),
        "details": details or {},
    }
    atomic_json_dump(progress, path)

def find_first_existing(candidates: Iterable[Path]) -> Optional[Path]:
    for path in candidates:
        if path.exists():
            return path
    return None

def recursive_find_prediction(root: Path, dataset: str, model_tokens: List[str], seed: int) -> Optional[Path]:
    if not root.exists():
        return None
    seed_token = f"seed_{seed}"
    matches = []
    for path in root.rglob("test_predictions.npz"):
        text = str(path).lower()
        if dataset.lower() in text and seed_token.lower() in text and all(t.lower() in text for t in model_tokens):
            matches.append(path)
    return sorted(matches, key=lambda p: len(str(p)))[0] if matches else None


In [4]:

# Cell 4 — Load exact frozen Step-5 subset, split and scaler

@dataclass
class FixedSubsetBundle:
    name: str
    feature_names: List[str]
    X_raw: np.ndarray
    src_ip: np.ndarray
    dst_ip: np.ndarray
    protocol: np.ndarray
    labels: np.ndarray
    attack_codes: np.ndarray
    attack_classes: np.ndarray
    subset_global_indices: np.ndarray
    subset_fingerprint: str

def read_step5_feature_names(dataset_name: str) -> List[str]:
    path = step5_dataset_dir(dataset_name) / "model_feature_names.csv"
    table = pd.read_csv(path)
    return table.sort_values("feature_index")["feature"].astype(str).tolist()

def preflight_step5(dataset_name: str) -> None:
    required = [
        subset_definition_dir(dataset_name) / "subset_indices_int32.npy",
        subset_definition_dir(dataset_name) / "subset_manifest.json",
        step5_dataset_dir(dataset_name) / "model_feature_names.csv",
        step5_shared_dir(dataset_name) / "split_indices.npz",
        step5_shared_dir(dataset_name) / "split_metadata.json",
        step5_shared_dir(dataset_name) / "split_overlap_audit.json",
        step5_shared_dir(dataset_name) / "scaler.joblib",
    ]
    for seed in MODEL_SEEDS:
        required.extend([
            step5_full_task_dir(dataset_name, seed) / "best.pt",
            step5_full_task_dir(dataset_name, seed) / "test_metrics.json",
            step5_full_task_dir(dataset_name, seed) / "test_predictions.npz",
        ])
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        message = "Thiếu Step-5 artifacts:\n" + "\n".join(missing)
        if STRICT_PREFLIGHT:
            raise FileNotFoundError(message)
        print("WARNING:", message)
    else:
        print("Preflight OK:", dataset_name)

def load_fixed_subset_bundle(dataset_name: str) -> FixedSubsetBundle:
    csv_path = Path(resolve_dataset_path(dataset_name))
    subset_idx = np.asarray(
        np.load(subset_definition_dir(dataset_name) / "subset_indices_int32.npy", mmap_mode="r"),
        dtype=np.int64,
    )
    manifest = load_json(subset_definition_dir(dataset_name) / "subset_manifest.json")
    if np.any(np.diff(subset_idx) < 0):
        raise RuntimeError("Step-5 subset indices are not sorted.")

    feature_names = read_step5_feature_names(dataset_name)
    context_cols = ["IPV4_SRC_ADDR", "IPV4_DST_ADDR", "PROTOCOL", "Label", "Attack"]
    usecols = list(dict.fromkeys(feature_names + context_cols))

    print(f"\nLoading frozen subset {dataset_name}: {len(subset_idx):,} rows")
    feature_chunks: List[np.ndarray] = []
    context_chunks: Dict[str, List[np.ndarray]] = {c: [] for c in context_cols}
    cursor = 0
    global_start = 0
    t0 = time.time()

    for chunk_no, df_chunk in enumerate(
        pd.read_csv(csv_path, usecols=usecols, chunksize=CSV_CHUNK_SIZE, low_memory=False),
        start=1,
    ):
        global_end = global_start + len(df_chunk)
        left = np.searchsorted(subset_idx, global_start, side="left")
        right = np.searchsorted(subset_idx, global_end, side="left")
        if right > left:
            local_rows = subset_idx[left:right] - global_start
            selected = df_chunk.iloc[local_rows]
            x = np.empty((len(selected), len(feature_names)), dtype=np.float32)
            for j, col in enumerate(feature_names):
                arr = pd.to_numeric(selected[col], errors="coerce").to_numpy(dtype=np.float32, copy=False)
                x[:, j] = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
            feature_chunks.append(x)
            for col in context_cols:
                context_chunks[col].append(selected[col].to_numpy(copy=True))
            cursor += len(selected)
        global_start = global_end
        if chunk_no % 10 == 0 or cursor == len(subset_idx):
            print(f"  chunks={chunk_no} | selected={cursor:,}/{len(subset_idx):,}")
        del df_chunk
        if cursor == len(subset_idx):
            break

    if cursor != len(subset_idx):
        raise RuntimeError(f"Loaded {cursor:,} rows; expected {len(subset_idx):,}.")

    X_raw = np.concatenate(feature_chunks, axis=0)
    context = {k: np.concatenate(v, axis=0) for k, v in context_chunks.items()}
    labels = pd.to_numeric(pd.Series(context["Label"]), errors="coerce").fillna(0).to_numpy(dtype=np.int8)
    attack_codes, attack_classes = pd.factorize(
        pd.Series(context["Attack"]).fillna("<MISSING_ATTACK>").astype(str),
        sort=False,
    )

    print(f"Loaded in {(time.time()-t0)/60:.1f} min | shape={X_raw.shape}")
    return FixedSubsetBundle(
        name=dataset_name,
        feature_names=feature_names,
        X_raw=X_raw,
        src_ip=context["IPV4_SRC_ADDR"],
        dst_ip=context["IPV4_DST_ADDR"],
        protocol=context["PROTOCOL"],
        labels=labels,
        attack_codes=attack_codes.astype(np.int16, copy=False),
        attack_classes=np.asarray(attack_classes, dtype=str),
        subset_global_indices=subset_idx,
        subset_fingerprint=str(manifest["subset_fingerprint"]),
    )

def load_frozen_step5_artifacts(bundle: FixedSubsetBundle):
    shared = step5_shared_dir(bundle.name)
    split_data = np.load(shared / "split_indices.npz")
    train_idx = np.asarray(split_data["train_idx"], dtype=np.int64)
    val_idx = np.asarray(split_data["val_idx"], dtype=np.int64)
    test_idx = np.asarray(split_data["test_idx"], dtype=np.int64)
    split_meta = load_json(shared / "split_metadata.json")
    overlap = load_json(shared / "split_overlap_audit.json")
    scaler = joblib.load(shared / "scaler.joblib")

    if split_meta.get("protocol_name") != PROTOCOL_NAME:
        raise RuntimeError("Step-5 protocol mismatch.")
    if max(
        float(overlap.get("validation_feature_overlap_rate", 1)),
        float(overlap.get("test_feature_overlap_rate", 1)),
    ) != 0.0:
        raise RuntimeError("Frozen Step-5 split has non-zero exact-feature overlap.")
    all_idx = np.concatenate([train_idx, val_idx, test_idx])
    if len(np.unique(all_idx)) != len(all_idx):
        raise RuntimeError("Frozen split row indices overlap.")

    if SMOKE_MODE:
        train_idx = train_idx[:SMOKE_MAX_ROWS_PER_SPLIT]
        val_idx = val_idx[:SMOKE_MAX_ROWS_PER_SPLIT]
        test_idx = test_idx[:SMOKE_MAX_ROWS_PER_SPLIT]

    print(
        f"Frozen split: train={len(train_idx):,}, val={len(val_idx):,}, "
        f"test={len(test_idx):,} | {split_meta['split_fingerprint']}"
    )
    return train_idx, val_idx, test_idx, scaler, split_meta


In [5]:
# Cell 5 — Metrics, validation threshold and operating points

# Cell 8 — Losses, threshold calibration, metrics, and score collection

def focal_bce_with_logits(
    logits: torch.Tensor,
    targets: torch.Tensor,
    alpha: float,
    gamma: float,
    pos_weight: torch.Tensor,
) -> torch.Tensor:
    bce = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="none",
        pos_weight=pos_weight,
    )
    probabilities = torch.sigmoid(logits)
    pt = torch.where(targets == 1, probabilities, 1 - probabilities)
    alpha_t = torch.where(
        targets == 1,
        torch.as_tensor(alpha, device=targets.device, dtype=targets.dtype),
        torch.as_tensor(1.0 - alpha, device=targets.device, dtype=targets.dtype),
    )
    return (alpha_t * (1 - pt).pow(gamma) * bce).mean()


def safe_divide(numerator: np.ndarray, denominator: np.ndarray) -> np.ndarray:
    result = np.zeros_like(numerator, dtype=np.float64)
    mask = denominator > 0
    result[mask] = numerator[mask] / denominator[mask]
    return result


def find_best_threshold_macro_f1_exact(scores: np.ndarray, labels: np.ndarray) -> Tuple[float, float]:
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)

    if len(scores) == 0 or len(np.unique(labels)) < 2:
        return 0.0, float(np.median(scores)) if len(scores) else 0.5

    order = np.argsort(scores)
    sorted_scores = scores[order]
    sorted_labels = labels[order]
    unique_ends = np.r_[np.where(np.diff(sorted_scores) != 0)[0], len(sorted_scores) - 1]

    cumulative_positive = np.cumsum(sorted_labels)[unique_ends]
    count_le = unique_ends + 1
    cumulative_negative = count_le - cumulative_positive
    total_positive = int(sorted_labels.sum())
    total_negative = int(len(sorted_labels) - total_positive)

    tp = total_positive - cumulative_positive
    fp = total_negative - cumulative_negative
    fn = cumulative_positive
    tn = cumulative_negative

    f1_positive = safe_divide(2 * tp, 2 * tp + fp + fn)
    f1_negative = safe_divide(2 * tn, 2 * tn + fn + fp)
    macro_f1 = 0.5 * (f1_positive + f1_negative)

    # Candidate: all samples predicted positive.
    all_positive_macro = 0.5 * (
        (2 * total_positive) / max(2 * total_positive + total_negative, 1)
    )

    best_index = int(np.argmax(macro_f1))
    best_f1 = float(macro_f1[best_index])
    best_threshold = float(sorted_scores[unique_ends[best_index]])

    if all_positive_macro > best_f1:
        best_f1 = float(all_positive_macro)
        best_threshold = float(np.nextafter(sorted_scores[0], -np.inf))

    return best_f1, best_threshold




def evaluate_scores(
    collected: Dict[str, np.ndarray],
    threshold: Optional[float] = None,
    score_sign: Optional[int] = None,
) -> Dict[str, Any]:
    raw_scores = collected["scores"].astype(np.float64, copy=False)
    labels = collected["labels"].astype(np.int64, copy=False)

    if len(np.unique(labels)) < 2:
        raise ValueError("Evaluation split must contain both benign and attack labels")

    raw_auc = roc_auc_score(labels, raw_scores)
    if score_sign is None:
        score_sign = -1 if raw_auc < 0.5 else 1
    oriented_scores = score_sign * raw_scores

    auc = roc_auc_score(labels, oriented_scores)
    pr_auc = average_precision_score(labels, oriented_scores)

    if threshold is None:
        f1_macro, threshold = find_best_threshold_macro_f1_exact(oriented_scores, labels)
    else:
        predictions = (oriented_scores > threshold).astype(np.int8)
        f1_macro = f1_score(labels, predictions, average="macro", zero_division=0)

    predictions = (oriented_scores > threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()

    return {
        "f1_macro": float(f1_macro),
        "auc_roc": float(auc),
        "pr_auc": float(pr_auc),
        "threshold": float(threshold),
        "score_sign": int(score_sign),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": float((tp + tn) / max(tp + tn + fp + fn, 1)),
        "precision": float(tp / max(tp + fp, 1)),
        "recall": float(tp / max(tp + fn, 1)),
        "specificity": float(tn / max(tn + fp, 1)),
        "balanced_accuracy": float(
            0.5 * (tp / max(tp + fn, 1) + tn / max(tn + fp, 1))
        ),
        "f1_attack": float(2 * tp / max(2 * tp + fp + fn, 1)),
        "f1_benign": float(2 * tn / max(2 * tn + fn + fp, 1)),
        "fp_rate": float(fp / max(fp + tn, 1)),
        "tp_rate": float(tp / max(tp + fn, 1)),
        "scores": oriented_scores.astype(np.float32),
        "predictions": predictions,
    }


def test_curve_operating_points(scores: np.ndarray, labels: np.ndarray):
    fpr,tpr,rt=roc_curve(labels,scores)
    precision,recall,pt=precision_recall_curve(labels,scores)
    rows=[]
    for target in TEST_CURVE_FPR_TARGETS:
        eligible=np.flatnonzero(fpr<=target+1e-15)
        chosen=eligible[np.argmax(tpr[eligible])] if len(eligible) else 0
        rows.append({'criterion':'TPR_at_test_FPR','target':target,'value':float(tpr[chosen]),'observed':float(fpr[chosen]),'curve_threshold':float(rt[chosen])})
    for target in TEST_CURVE_RECALL_TARGETS:
        eligible=np.flatnonzero(recall>=target-1e-15)
        chosen=eligible[np.argmax(precision[eligible])] if len(eligible) else int(np.argmax(recall))
        thr=float('nan') if chosen>=len(pt) else float(pt[chosen])
        rows.append({'criterion':'Precision_at_test_recall','target':target,'value':float(precision[chosen]),'observed':float(recall[chosen]),'curve_threshold':thr})
    return rows

In [6]:

# Cell 6 — Graph windows and relation-aware THAT-IDS variants

@dataclass(frozen=True)
class VariantConfig:
    name: str
    relation_mode: str  # both | ip_ip_only | ip_protocol_only
    drop_protocol_numeric: bool = False

FULL_VARIANT = VariantConfig("full", "both", False)
RELATION_VARIANTS = {
    "ip_ip_only": VariantConfig("ip_ip_only", "ip_ip_only", False),
    "ip_protocol_only": VariantConfig("ip_protocol_only", "ip_protocol_only", False),
    "both_no_protocol_numeric": VariantConfig("both_no_protocol_numeric", "both", True),
}

@dataclass
class GraphWindow:
    edge_feat: torch.Tensor
    labels: torch.Tensor
    src: torch.Tensor
    dst: torch.Tensor
    proto_local: torch.Tensor
    n_ip: int
    n_proto: int
    row_indices: np.ndarray

def protocol_feature_mask(feature_names: List[str], drop_protocol_numeric: bool) -> np.ndarray:
    mask = np.ones(len(feature_names), dtype=bool)
    if not drop_protocol_numeric:
        return mask
    drop_names = {"PROTOCOL", "L7_PROTO"}
    found = []
    for i, name in enumerate(feature_names):
        if str(name).upper() in drop_names:
            mask[i] = False
            found.append(str(name))
    if not found:
        print("WARNING: no PROTOCOL/L7_PROTO numerical columns were found.")
    else:
        print("Dropping numerical protocol fields:", found)
    if not mask.any():
        raise RuntimeError("Feature mask removed every numerical feature.")
    return mask

def permute_split_for_partition(split_idx: np.ndarray, partition_seed: int, split_offset: int) -> np.ndarray:
    split_idx = np.asarray(split_idx, dtype=np.int64)
    if partition_seed == 0:
        return split_idx.copy()
    rng = np.random.default_rng(int(partition_seed) * 1009 + int(split_offset))
    return split_idx[rng.permutation(len(split_idx))]

def build_graph_window(
    bundle: FixedSubsetBundle,
    row_idx: np.ndarray,
    scaler: MinMaxScaler,
) -> GraphWindow:
    raw_feat = bundle.X_raw[row_idx]
    scaled_feat = scaler.transform(raw_feat)
    scaled_feat = np.clip(scaled_feat, CLIP_MIN, CLIP_MAX).astype(np.float32, copy=False)

    src_values = pd.Series(bundle.src_ip[row_idx], copy=False).fillna("<MISSING_IP>").astype(str).to_numpy()
    dst_values = pd.Series(bundle.dst_ip[row_idx], copy=False).fillna("<MISSING_IP>").astype(str).to_numpy()
    proto_values = pd.Series(bundle.protocol[row_idx], copy=False).fillna("<MISSING_PROTOCOL>").astype(str).to_numpy()

    combined_ip = np.concatenate([src_values, dst_values])
    ip_codes, _ = pd.factorize(combined_ip, sort=False)
    n = len(row_idx)
    src = ip_codes[:n].astype(np.int64, copy=False)
    dst = ip_codes[n:].astype(np.int64, copy=False)
    proto_local, proto_uniques = pd.factorize(proto_values, sort=False)
    proto_local = proto_local.astype(np.int64, copy=False)

    return GraphWindow(
        edge_feat=torch.from_numpy(scaled_feat),
        labels=torch.from_numpy(bundle.labels[row_idx].astype(np.int64, copy=False)),
        src=torch.from_numpy(src),
        dst=torch.from_numpy(dst),
        proto_local=torch.from_numpy(proto_local),
        n_ip=int(ip_codes.max() + 1) if len(ip_codes) else 0,
        n_proto=int(len(proto_uniques)),
        row_indices=np.asarray(row_idx, dtype=np.int64),
    )

def prepare_graph_windows(
    bundle: FixedSubsetBundle,
    split_idx: np.ndarray,
    scaler: MinMaxScaler,
    description: str,
) -> List[GraphWindow]:
    windows = []
    n_windows = math.ceil(len(split_idx) / WINDOW_SIZE)
    t0 = time.time()
    for w in range(n_windows):
        idx = split_idx[w * WINDOW_SIZE:(w + 1) * WINDOW_SIZE]
        if len(idx):
            windows.append(build_graph_window(bundle, idx, scaler))
        if (w + 1) % 500 == 0 or w + 1 == n_windows:
            print(f"    {description}: {w + 1}/{n_windows}")
    print(f"  {description}: {len(windows)} batches in {(time.time()-t0)/60:.1f} min")
    return windows

class RelationAttentionEncoder(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, n_relations: int, dropout: float = 0.10):
        super().__init__()
        self.out_dim = out_dim
        self.n_relations = n_relations
        self.proj = nn.Linear(in_dim, out_dim)
        self.edge_attention = nn.ModuleList([
            nn.Linear(out_dim * 2, 1) for _ in range(n_relations)
        ])
        self.semantic_attention = nn.Linear(out_dim, 1)
        self.dropout = nn.Dropout(dropout)
        self.norms = nn.ModuleList([nn.LayerNorm(out_dim) for _ in range(n_relations)])
        self.last_semantic_weights: Optional[torch.Tensor] = None
        self.last_edge_attention_means: Optional[List[float]] = None

    def aggregate_relation(self, h, edge_src, edge_dst, relation_idx):
        if edge_src.numel() == 0:
            self._current_edge_means.append(float("nan"))
            return h
        src_h = h[edge_src]
        dst_h = h[edge_dst]
        alpha = torch.sigmoid(
            self.edge_attention[relation_idx](torch.cat([src_h, dst_h], dim=1))
        )
        self._current_edge_means.append(float(alpha.detach().mean().item()))
        messages = src_h * alpha
        out = torch.zeros_like(h)
        weight_sum = torch.zeros(h.size(0), 1, device=h.device, dtype=h.dtype)
        out.scatter_add_(0, edge_dst.unsqueeze(1).expand(-1, self.out_dim), messages)
        weight_sum.scatter_add_(0, edge_dst.unsqueeze(1), alpha)
        out = out / weight_sum.clamp(min=1.0)
        return self.norms[relation_idx](h + self.dropout(out))

    def forward(self, node_features, relations):
        if len(relations) != self.n_relations:
            raise ValueError(f"Expected {self.n_relations} relations, got {len(relations)}")
        self._current_edge_means = []
        h = F.relu(self.proj(node_features))
        outputs = [
            self.aggregate_relation(h, src, dst, relation_idx)
            for relation_idx, (src, dst) in enumerate(relations)
        ]
        self.last_edge_attention_means = list(self._current_edge_means)
        if len(outputs) == 1:
            weights = torch.ones(
                (1, h.size(0), 1), device=h.device, dtype=h.dtype
            )
            self.last_semantic_weights = weights.detach()
            return outputs[0]
        stacked = torch.stack(outputs, dim=0)
        weights = torch.softmax(self.semantic_attention(stacked), dim=0)
        self.last_semantic_weights = weights.detach()
        return (weights * stacked).sum(dim=0)

class HybridDetector(nn.Module):
    def __init__(self, input_dim, flow_dim, hidden_dim, latent_dim,
                 ctx_recon_weight=CTX_RECON_WEIGHT):
        super().__init__()
        self.flow_dim = flow_dim
        self.ctx_recon_weight = ctx_recon_weight
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(ENC_DROPOUT),
            nn.Linear(hidden_dim, latent_dim), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, input_dim)
        )
        self.classifier = nn.Sequential(
            nn.Linear(input_dim + latent_dim, hidden_dim), nn.ReLU(), nn.Dropout(CLS_DROPOUT),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x):
        z = self.encoder(x)
        reconstruction = self.decoder(z)
        logits = self.classifier(torch.cat([x, z], dim=1)).squeeze(1)
        return z, reconstruction, logits

    def reconstruction_error(self, x, reconstruction):
        err = (reconstruction - x) ** 2
        flow_err = err[:, :self.flow_dim].mean(dim=1)
        context_err = err[:, self.flow_dim:].mean(dim=1)
        return flow_err + self.ctx_recon_weight * context_err

class THATIDSRelationModel(nn.Module):
    def __init__(self, feature_names: List[str], variant: VariantConfig):
        super().__init__()
        self.original_feature_names = list(feature_names)
        self.variant = variant
        mask = protocol_feature_mask(feature_names, variant.drop_protocol_numeric)
        self.register_buffer("feature_mask", torch.from_numpy(mask), persistent=False)
        self.active_feature_names = [n for n, keep in zip(feature_names, mask) if keep]
        self.flow_dim = len(self.active_feature_names)
        n_relations = 2 if variant.relation_mode == "both" else 1

        self.graph_encoder = RelationAttentionEncoder(
            self.flow_dim, HIDDEN_DIM, n_relations, GRAPH_DROPOUT
        )
        self.context_gate = nn.Parameter(torch.tensor(float(INIT_GATE_LOGIT)))
        self.detector = HybridDetector(
            self.flow_dim + HIDDEN_DIM, self.flow_dim, AE_HIDDEN, LATENT_DIM
        )

    def gate_value(self) -> float:
        return float(torch.sigmoid(self.context_gate.detach()).item())

    def prepare_flow_input(self, window: GraphWindow, device: torch.device):
        all_features = window.edge_feat.to(device, non_blocking=True)
        mask = self.feature_mask.to(device)
        flow_features = all_features[:, mask]
        labels = window.labels.to(device, non_blocking=True)
        src = window.src.to(device, non_blocking=True)
        dst = window.dst.to(device, non_blocking=True)
        proto_local = window.proto_local.to(device, non_blocking=True)
        proto = proto_local + window.n_ip
        n_nodes = window.n_ip + window.n_proto
        n_flows = flow_features.size(0)

        node_features = torch.zeros(
            n_nodes, flow_features.size(1), device=device, dtype=flow_features.dtype
        )
        counts = torch.zeros(n_nodes, 1, device=device, dtype=flow_features.dtype)
        ones = torch.ones(n_flows, 1, device=device, dtype=flow_features.dtype)
        for node_ids in (src, dst, proto):
            node_features.scatter_add_(
                0, node_ids.unsqueeze(1).expand(-1, flow_features.size(1)), flow_features
            )
            counts.scatter_add_(0, node_ids.unsqueeze(1), ones)
        node_features = node_features / counts.clamp(min=1.0)

        comm_src = torch.cat([src, dst], dim=0)
        comm_dst = torch.cat([dst, src], dim=0)
        proto_src = torch.cat([src, proto], dim=0)
        proto_dst = torch.cat([proto, src], dim=0)

        if self.variant.relation_mode == "both":
            relations = [(comm_src, comm_dst), (proto_src, proto_dst)]
            readout_nodes = (src, dst, proto)
        elif self.variant.relation_mode == "ip_ip_only":
            relations = [(comm_src, comm_dst)]
            readout_nodes = (src, dst)
        elif self.variant.relation_mode == "ip_protocol_only":
            relations = [(proto_src, proto_dst)]
            readout_nodes = (src, proto)
        else:
            raise ValueError(self.variant.relation_mode)

        node_embeddings = self.graph_encoder(node_features, relations)
        gate = torch.sigmoid(self.context_gate)
        context = gate * sum(node_embeddings[node_ids] for node_ids in readout_nodes) / len(readout_nodes)
        detector_input = torch.cat([flow_features, context], dim=1)
        return detector_input, labels

def load_step5_full_model(dataset_name: str, seed: int, feature_names: List[str]):
    checkpoint = load_torch_checkpoint(
        step5_full_task_dir(dataset_name, seed) / "best.pt", DEVICE
    )
    model = THATIDSRelationModel(feature_names, FULL_VARIANT).to(DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.eval()
    return model, checkpoint


In [7]:

# Cell 7 — Training, checkpoint/save point and task execution

def focal_bce_with_logits(logits, targets, alpha, gamma, pos_weight):
    bce = F.binary_cross_entropy_with_logits(
        logits, targets, reduction="none", pos_weight=pos_weight
    )
    probabilities = torch.sigmoid(logits)
    pt = torch.where(targets == 1, probabilities, 1 - probabilities)
    alpha_t = torch.where(
        targets == 1,
        torch.as_tensor(alpha, device=targets.device, dtype=targets.dtype),
        torch.as_tensor(1.0-alpha, device=targets.device, dtype=targets.dtype),
    )
    return (alpha_t * (1-pt).pow(gamma) * bce).mean()

def load_history(path: Path) -> List[Dict[str, Any]]:
    return pd.read_csv(path).to_dict(orient="records") if path.exists() else []

def save_history(rows: List[Dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".csv.tmp")
    pd.DataFrame(rows).to_csv(tmp, index=False)
    os.replace(tmp, path)

@torch.no_grad()
def collect_model_components(model, windows, device=DEVICE):
    model.eval()
    probabilities, reconstruction_errors, labels_all = [], [], []
    row_indices = []
    semantic_rows, edge_rows = [], []

    for batch_no, window in enumerate(windows):
        detector_input, labels = model.prepare_flow_input(window, device)
        _, reconstruction, logits = model.detector(detector_input)
        prob = torch.sigmoid(logits)
        rec = model.detector.reconstruction_error(detector_input, reconstruction)

        probabilities.append(prob.cpu().numpy().astype(np.float32))
        reconstruction_errors.append(rec.cpu().numpy().astype(np.float32))
        labels_all.append(labels.cpu().numpy().astype(np.int8))
        row_indices.append(window.row_indices.astype(np.int64))

        weights = model.graph_encoder.last_semantic_weights
        if weights is not None:
            mean_weights = weights.mean(dim=1).squeeze(-1).cpu().numpy()
            for relation_index, value in enumerate(mean_weights):
                semantic_rows.append({
                    "batch": batch_no,
                    "relation_index": relation_index,
                    "weight": float(value),
                })
        edge_means = model.graph_encoder.last_edge_attention_means or []
        for relation_index, value in enumerate(edge_means):
            edge_rows.append({
                "batch": batch_no,
                "relation_index": relation_index,
                "mean_edge_attention": float(value),
            })

    return {
        "probabilities": np.concatenate(probabilities),
        "reconstruction_errors": np.concatenate(reconstruction_errors),
        "labels": np.concatenate(labels_all).astype(np.int64),
        "row_indices": np.concatenate(row_indices).astype(np.int64),
        "semantic_attention": pd.DataFrame(semantic_rows),
        "edge_attention": pd.DataFrame(edge_rows),
    }

def hybrid_scores(probabilities: np.ndarray, reconstruction_errors: np.ndarray, beta: float) -> np.ndarray:
    return (
        float(beta) * probabilities.astype(np.float64)
        + (1.0 - float(beta)) * np.log1p(reconstruction_errors.astype(np.float64))
    )

def train_one_epoch(model, windows, optimizer, positive_class_weight):
    model.train()
    totals = {"loss": 0.0, "reconstruction_loss": 0.0, "classification_loss": 0.0}
    batches = 0
    pos_weight = torch.tensor([positive_class_weight], device=DEVICE, dtype=torch.float32)
    for window in windows:
        detector_input, labels = model.prepare_flow_input(window, DEVICE)
        _, reconstruction, logits = model.detector(detector_input)
        benign_mask = labels == 0
        reconstruction_loss = torch.zeros((), device=DEVICE)
        if benign_mask.any():
            reconstruction_loss = model.detector.reconstruction_error(
                detector_input[benign_mask], reconstruction[benign_mask]
            ).mean()
        classification_loss = focal_bce_with_logits(
            logits, labels.float(), FOCAL_ALPHA, FOCAL_GAMMA, pos_weight
        )
        loss = LAMBDA_RECON * reconstruction_loss + LAMBDA_CLS * classification_loss
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()
        totals["loss"] += float(loss.detach().item())
        totals["reconstruction_loss"] += float(reconstruction_loss.detach().item())
        totals["classification_loss"] += float(classification_loss.detach().item())
        batches += 1
    return {k: v / max(batches, 1) for k, v in totals.items()}

def evaluate_component_bundle(collected, threshold=None, score_sign=None):
    scores = hybrid_scores(
        collected["probabilities"], collected["reconstruction_errors"], SCORE_CLS_WEIGHT
    )
    result = evaluate_scores(
        {"scores": scores, "labels": collected["labels"]},
        threshold=threshold,
        score_sign=score_sign,
    )
    return result, scores

def run_training_task(
    bundle: FixedSubsetBundle,
    variant: VariantConfig,
    model_seed: int,
    train_windows: List[GraphWindow],
    val_windows: List[GraphWindow],
    test_windows: List[GraphWindow],
    positive_class_weight: float,
    task_dir: Path,
    experiment_payload: Dict[str, Any],
) -> Dict[str, Any]:
    task_dir.mkdir(parents=True, exist_ok=True)
    completed_path = task_dir / "completed.json"
    best_path = task_dir / "best.pt"
    last_path = task_dir / "last.pt"
    history_path = task_dir / "history.csv"
    result_path = task_dir / "test_metrics.json"
    val_prediction_path = task_dir / "validation_predictions.npz"
    test_prediction_path = task_dir / "test_predictions.npz"
    config_path = task_dir / "config.json"
    fingerprint_path = task_dir / "config_fingerprint.txt"

    payload = {
        **experiment_payload,
        "dataset": bundle.name,
        "subset_fingerprint": bundle.subset_fingerprint,
        "model_seed": int(model_seed),
        "variant": asdict(variant),
        "feature_names": bundle.feature_names,
        "window_size": WINDOW_SIZE,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "optimizer": "AdamW",
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
    }
    fingerprint = config_fingerprint(payload)

    if completed_path.exists():
        completed = load_json(completed_path)
        if (
            completed.get("config_fingerprint") == fingerprint
            and result_path.exists() and val_prediction_path.exists()
            and test_prediction_path.exists()
        ):
            print("  SKIP completed:", task_dir)
            return load_json(result_path)
        if completed.get("config_fingerprint") != fingerprint:
            raise RuntimeError(f"Fingerprint mismatch in completed task: {task_dir}")
        completed_path.unlink(missing_ok=True)

    if fingerprint_path.exists() and fingerprint_path.read_text().strip() != fingerprint:
        raise RuntimeError(
            f"Config fingerprint mismatch: {task_dir}. "
            "Use a new STEP8_EXPERIMENT_TAG or remove this task directory."
        )
    atomic_json_dump(payload, config_path)
    fingerprint_path.write_text(fingerprint, encoding="utf-8")

    seed_everything(model_seed)
    model = THATIDSRelationModel(bundle.feature_names, variant).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

    start_epoch = 1
    best_val_f1 = -1.0
    best_epoch = -1
    best_threshold = 0.5
    best_score_sign = 1
    patience_counter = 0
    history = load_history(history_path)

    if last_path.exists():
        checkpoint = load_torch_checkpoint(last_path, DEVICE)
        if checkpoint.get("config_fingerprint") != fingerprint:
            raise RuntimeError(f"Checkpoint fingerprint mismatch: {last_path}")
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        start_epoch = int(checkpoint["epoch"]) + 1
        best_val_f1 = float(checkpoint["best_val_f1"])
        best_epoch = int(checkpoint["best_epoch"])
        best_threshold = float(checkpoint["best_threshold"])
        best_score_sign = int(checkpoint.get("best_score_sign", 1))
        patience_counter = int(checkpoint["patience_counter"])
        restore_rng_state(checkpoint.get("rng_state"))
        print(f"  RESUME {variant.name} seed={model_seed} from epoch {start_epoch}")
    else:
        print(f"  START {variant.name} seed={model_seed} | params={parameter_count:,}")

    task_start = time.time()
    for epoch in range(start_epoch, MAX_EPOCHS + 1) if patience_counter < PATIENCE else []:
        epoch_start = time.time()
        train_metrics = train_one_epoch(
            model, train_windows, optimizer, positive_class_weight
        )
        val_collected = collect_model_components(model, val_windows)
        val_metrics, _ = evaluate_component_bundle(val_collected)

        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_reconstruction_loss": train_metrics["reconstruction_loss"],
            "train_classification_loss": train_metrics["classification_loss"],
            "val_f1_macro": val_metrics["f1_macro"],
            "val_auc_roc": val_metrics["auc_roc"],
            "val_pr_auc": val_metrics["pr_auc"],
            "val_threshold": val_metrics["threshold"],
            "gate": model.gate_value(),
            "epoch_seconds": time.time() - epoch_start,
        }
        history = [h for h in history if int(h["epoch"]) != epoch]
        history.append(row)
        history.sort(key=lambda x: int(x["epoch"]))
        save_history(history, history_path)

        improved = val_metrics["f1_macro"] > best_val_f1
        if improved:
            best_val_f1 = float(val_metrics["f1_macro"])
            best_epoch = epoch
            best_threshold = float(val_metrics["threshold"])
            best_score_sign = int(val_metrics["score_sign"])
            patience_counter = 0
            atomic_torch_save({
                "model_state_dict": model.state_dict(),
                "epoch": epoch,
                "best_val_f1": best_val_f1,
                "best_threshold": best_threshold,
                "best_score_sign": best_score_sign,
                "variant": asdict(variant),
                "config_fingerprint": fingerprint,
            }, best_path)
        else:
            patience_counter += 1

        if epoch % SAVE_LAST_EVERY == 0 or improved or epoch == MAX_EPOCHS:
            atomic_torch_save({
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "epoch": epoch,
                "best_val_f1": best_val_f1,
                "best_epoch": best_epoch,
                "best_threshold": best_threshold,
                "best_score_sign": best_score_sign,
                "patience_counter": patience_counter,
                "rng_state": capture_rng_state(),
                "variant": asdict(variant),
                "config_fingerprint": fingerprint,
            }, last_path)

        print(
            f"    Ep {epoch:03d}/{MAX_EPOCHS} | loss={train_metrics['loss']:.5f} | "
            f"valF1={val_metrics['f1_macro']:.4f} | PR={val_metrics['pr_auc']:.4f} | "
            f"thr={val_metrics['threshold']:.4f} | gate={model.gate_value():.4f} | "
            f"pat={patience_counter}/{PATIENCE} | {time.time()-epoch_start:.0f}s"
        )
        del val_collected
        if patience_counter >= PATIENCE:
            print("    Early stop at epoch", epoch, "| best epoch", best_epoch)
            break

    if not best_path.exists():
        raise RuntimeError("Best checkpoint was not created: " + str(best_path))
    best = load_torch_checkpoint(best_path, DEVICE)
    model.load_state_dict(best["model_state_dict"])
    model.eval()

    val_collected = collect_model_components(model, val_windows)
    test_collected = collect_model_components(model, test_windows)
    val_metrics, val_scores = evaluate_component_bundle(val_collected)
    test_metrics, test_scores = evaluate_component_bundle(
        test_collected, threshold=best_threshold, score_sign=best_score_sign
    )

    test_rows = test_collected["row_indices"]
    attack_codes = bundle.attack_codes[test_rows]
    per_attack = []
    for code, attack_name in enumerate(bundle.attack_classes):
        mask = (attack_codes == code) & (test_collected["labels"] == 1)
        count = int(mask.sum())
        if count:
            detected = int(test_metrics["predictions"][mask].sum())
            per_attack.append({
                "attack": str(attack_name),
                "count": count,
                "detected": detected,
                "recall": float(detected / count),
            })

    relation_names = (
        ["communication", "protocol"]
        if variant.relation_mode == "both"
        else ["communication"] if variant.relation_mode == "ip_ip_only"
        else ["protocol"]
    )
    semantic = test_collected["semantic_attention"]
    edge_attn = test_collected["edge_attention"]
    attention_summary = {
        "gate_final": model.gate_value(),
        "relation_names": relation_names,
        "semantic_attention": {},
        "edge_attention": {},
    }
    for i, name in enumerate(relation_names):
        s = semantic.loc[semantic["relation_index"] == i, "weight"] if not semantic.empty else pd.Series(dtype=float)
        e = edge_attn.loc[edge_attn["relation_index"] == i, "mean_edge_attention"] if not edge_attn.empty else pd.Series(dtype=float)
        attention_summary["semantic_attention"][name] = {
            "mean": float(s.mean()) if len(s) else float("nan"),
            "std": float(s.std(ddof=1)) if len(s) > 1 else 0.0,
            "n_batches": int(len(s)),
        }
        attention_summary["edge_attention"][name] = {
            "mean": float(e.mean()) if len(e) else float("nan"),
            "std": float(e.std(ddof=1)) if len(e) > 1 else 0.0,
            "n_batches": int(len(e)),
        }
    atomic_json_dump(attention_summary, task_dir / "attention_gate_summary.json")

    np.savez_compressed(
        val_prediction_path,
        scores=val_scores.astype(np.float32),
        labels=val_collected["labels"].astype(np.int8),
        probabilities=val_collected["probabilities"].astype(np.float32),
        reconstruction_errors=val_collected["reconstruction_errors"].astype(np.float32),
        row_indices=val_collected["row_indices"].astype(np.int64),
    )
    np.savez_compressed(
        test_prediction_path,
        scores=test_scores.astype(np.float32),
        predictions=test_metrics["predictions"].astype(np.int8),
        labels=test_collected["labels"].astype(np.int8),
        probabilities=test_collected["probabilities"].astype(np.float32),
        reconstruction_errors=test_collected["reconstruction_errors"].astype(np.float32),
        row_indices=test_collected["row_indices"].astype(np.int64),
        attack_codes=attack_codes.astype(np.int16),
        attack_classes=bundle.attack_classes.astype(str),
    )

    result = {
        "dataset": bundle.name,
        "variant": variant.name,
        "model_seed": model_seed,
        "parameters": parameter_count,
        "best_epoch": best_epoch,
        "validation": {
            k: v for k, v in val_metrics.items()
            if k not in {"scores", "predictions"}
        },
        "test": {
            k: v for k, v in test_metrics.items()
            if k not in {"scores", "predictions"}
        },
        "gate_final": model.gate_value(),
        "per_attack": per_attack,
        "attention_summary": attention_summary,
        "elapsed_seconds_current_session": time.time() - task_start,
        "elapsed_seconds_history": float(sum(float(r.get("epoch_seconds", 0)) for r in history)),
        "config_fingerprint": fingerprint,
        **experiment_payload,
    }
    atomic_json_dump(result, result_path)
    atomic_json_dump({
        "completed": True,
        "completed_at_unix": time.time(),
        "dataset": bundle.name,
        "variant": variant.name,
        "model_seed": model_seed,
        "config_fingerprint": fingerprint,
    }, completed_path)

    print(
        f"  DONE {variant.name} seed={model_seed} | "
        f"F1={result['test']['f1_macro']:.4f} | "
        f"PR={result['test']['pr_auc']:.4f} | "
        f"FPR={100*result['test']['fp_rate']:.4f}%"
    )
    del model, optimizer, val_collected, test_collected
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


In [8]:

# Cell 8 — Audit phase: duplicate conflicts, Table XI, gate/attention, β sensitivity, bootstrap

def locate_full_duplicate_audit(dataset_name: str) -> Optional[Path]:
    candidates = [
        STEP4_SHARED_CACHE_ROOT / dataset_name / "duplicate_audit_summary.json",
        STEP4_ROOT / "pilot" / dataset_name / "duplicate_audit_summary.json",
        STEP4_ROOT / "paper" / dataset_name / "duplicate_audit_summary.json",
        STEP4_ROOT / dataset_name / "duplicate_audit_summary.json",
    ]
    found = find_first_existing(candidates)
    if found:
        return found
    matches = list(DRIVE_RUN_ROOT.rglob(f"{dataset_name}/duplicate_audit_summary.json"))
    return sorted(matches, key=lambda p: len(str(p)))[0] if matches else None

def verify_hash_groups_sample(bundle, group_inverse: np.ndarray, group_counts: np.ndarray,
                              max_groups: int, seed: int = 2027) -> Dict[str, Any]:
    duplicate_groups = np.flatnonzero(np.asarray(group_counts) > 1)
    if HASH_VERIFY_MODE == "none" or len(duplicate_groups) == 0:
        return {"mode": HASH_VERIFY_MODE, "groups_checked": 0, "non_equal_hash_groups": None}
    rng = np.random.default_rng(seed)
    if HASH_VERIFY_MODE == "sample" and len(duplicate_groups) > max_groups:
        groups = rng.choice(duplicate_groups, size=max_groups, replace=False)
    else:
        groups = duplicate_groups

    selected = set(int(x) for x in groups)
    first_rows: Dict[int, np.ndarray] = {}
    non_equal = 0
    checked = 0
    for row, group in enumerate(np.asarray(group_inverse)):
        g = int(group)
        if g not in selected:
            continue
        x = bundle.X_raw[row]
        if g not in first_rows:
            first_rows[g] = x.copy()
        elif not np.array_equal(first_rows[g], x, equal_nan=True):
            non_equal += 1
            selected.remove(g)
        if not selected:
            break
    checked = len(groups)
    return {
        "mode": HASH_VERIFY_MODE,
        "groups_checked": int(checked),
        "non_equal_hash_groups": int(non_equal),
        "sample_seed": int(seed),
    }

def run_duplicate_conflict_audit(dataset_name: str) -> Dict[str, Any]:
    path = locate_full_duplicate_audit(dataset_name)
    if path is None:
        raise FileNotFoundError(
            f"Không tìm thấy Step-4 duplicate audit cho {dataset_name}. "
            "Hãy chạy lại Step 4 audit trước."
        )
    summary = load_json(path)
    out = {
        "dataset": dataset_name,
        "source": str(path),
        "n_rows": summary.get("n_rows"),
        "n_features": summary.get("n_features"),
        "n_unique_feature_groups": summary.get("n_unique_feature_groups"),
        "maximum_group_size": summary.get("maximum_group_size"),
        "binary_label_conflicting_groups": summary.get("binary_label_conflicting_groups"),
        "rows_in_binary_label_conflicting_groups": summary.get("rows_in_binary_label_conflicting_groups"),
        "attack_name_conflicting_groups": summary.get("attack_name_conflicting_groups"),
        "rows_in_attack_name_conflicting_groups": summary.get("rows_in_attack_name_conflicting_groups"),
        "repeated_rows_rate": summary.get("repeated_rows_rate"),
    }
    n_groups = max(int(out["n_unique_feature_groups"] or 0), 1)
    n_rows = max(int(out["n_rows"] or 0), 1)
    out["binary_conflicting_group_rate"] = (
        float(out["binary_label_conflicting_groups"] or 0) / n_groups
    )
    out["binary_conflicting_row_rate"] = (
        float(out["rows_in_binary_label_conflicting_groups"] or 0) / n_rows
    )
    out["attack_conflicting_group_rate"] = (
        float(out["attack_name_conflicting_groups"] or 0) / n_groups
    )
    return out

def hash_protocol_manifest(dataset_name: str) -> Dict[str, Any]:
    feature_names = read_step5_feature_names(dataset_name)
    return {
        "dataset": dataset_name,
        "group_key_scope": "39-dimensional local numerical feature vector before scaling",
        "dtype_before_hash": "float32",
        "missing_and_infinite_handling": "NaN, +inf and -inf converted to 0.0 while loading",
        "hash_function": "pandas.util.hash_pandas_object(frame, index=False)",
        "hash_output": "uint64",
        "column_order": feature_names,
        "byte_order_note": (
            "The implementation delegates numeric hashing to the installed pandas version; "
            "it does not manually serialize row bytes."
        ),
        "zero_handling_note": (
            "No explicit canonicalization of +0.0 and -0.0 is applied before pandas hashing."
        ),
        "collision_verification": HASH_VERIFY_MODE,
    }

def prediction_paths_for_table_xi(dataset: str, seed: int) -> Dict[str, Optional[Path]]:
    return {
        "LightGBM": step7_task_dir(dataset, "lightgbm", seed) / "test_predictions.npz",
        "Full": step5_full_task_dir(dataset, seed) / "test_predictions.npz",
        "E-GraphSAGE": recursive_find_prediction(
            STEP6_ROOT, dataset, ["egraph", "sage"], seed
        ),
    }

def aggregate_operating_points(dataset_name: str) -> pd.DataFrame:
    rows = []
    for seed in MODEL_SEEDS:
        paths = prediction_paths_for_table_xi(dataset_name, seed)
        for model_name, path in paths.items():
            if path is None or not path.exists():
                print("Missing Table-XI predictions:", dataset_name, model_name, seed, path)
                continue
            data = np.load(path, allow_pickle=True)
            scores = np.asarray(data["scores"], dtype=np.float64)
            labels = np.asarray(data["labels"], dtype=np.int64)
            for point in test_curve_operating_points(scores, labels):
                rows.append({
                    "dataset": dataset_name,
                    "model": model_name,
                    "seed": seed,
                    **point,
                    "source": str(path),
                })
    return pd.DataFrame(rows)

def summarize_operating_points(all_points: pd.DataFrame) -> pd.DataFrame:
    if all_points.empty:
        return pd.DataFrame()
    keys = ["dataset", "model", "criterion", "target"]
    return (
        all_points.groupby(keys, as_index=False)
        .agg(
            value_mean=("value", "mean"),
            value_std=("value", lambda x: float(np.std(x, ddof=1)) if len(x) > 1 else 0.0),
            n_seeds=("seed", "nunique"),
        )
    )

def run_checkpoint_diagnostics(dataset_name: str, bundle, train_idx, val_idx, test_idx, scaler):
    rows_gate, rows_attention, rows_beta, rows_hard = [], [], [], []
    val_windows = prepare_graph_windows(bundle, val_idx, scaler, "Diagnostic validation")
    test_windows = prepare_graph_windows(bundle, test_idx, scaler, "Diagnostic test")

    for seed in MODEL_SEEDS:
        model, _ = load_step5_full_model(dataset_name, seed, bundle.feature_names)
        val = collect_model_components(model, val_windows)
        test = collect_model_components(model, test_windows)
        rows_gate.append({
            "dataset": dataset_name,
            "seed": seed,
            "gate": model.gate_value(),
        })
        for split_name, collected in [("validation", val), ("test", test)]:
            sem = collected["semantic_attention"]
            edge = collected["edge_attention"]
            for relation_idx, relation_name in enumerate(["communication", "protocol"]):
                s = sem.loc[sem["relation_index"] == relation_idx, "weight"] if not sem.empty else pd.Series(dtype=float)
                e = edge.loc[edge["relation_index"] == relation_idx, "mean_edge_attention"] if not edge.empty else pd.Series(dtype=float)
                rows_attention.append({
                    "dataset": dataset_name,
                    "seed": seed,
                    "split": split_name,
                    "relation": relation_name,
                    "semantic_weight_mean": float(s.mean()) if len(s) else np.nan,
                    "semantic_weight_std_across_batches": float(s.std(ddof=1)) if len(s) > 1 else 0.0,
                    "edge_attention_mean": float(e.mean()) if len(e) else np.nan,
                    "edge_attention_std_across_batches": float(e.std(ddof=1)) if len(e) > 1 else 0.0,
                    "n_batches": int(max(len(s), len(e))),
                })

        for beta in BETA_VALUES:
            val_scores = hybrid_scores(val["probabilities"], val["reconstruction_errors"], beta)
            val_eval = evaluate_scores({"scores": val_scores, "labels": val["labels"]})
            test_scores = hybrid_scores(test["probabilities"], test["reconstruction_errors"], beta)
            test_eval = evaluate_scores(
                {"scores": test_scores, "labels": test["labels"]},
                threshold=float(val_eval["threshold"]),
                score_sign=int(val_eval["score_sign"]),
            )
            rows_beta.append({
                "dataset": dataset_name,
                "seed": seed,
                "beta_classifier_weight": beta,
                "validation_threshold": val_eval["threshold"],
                "validation_f1_macro": val_eval["f1_macro"],
                "test_f1_macro": test_eval["f1_macro"],
                "test_auc_roc": test_eval["auc_roc"],
                "test_pr_auc": test_eval["pr_auc"],
                "test_fp_rate": test_eval["fp_rate"],
                "test_tp_rate": test_eval["tp_rate"],
            })

        base_path = step5_full_task_dir(dataset_name, seed) / "test_metrics.json"
        if base_path.exists():
            result = load_json(base_path)
            for item in result.get("per_attack", []):
                rows_hard.append({
                    "dataset": dataset_name,
                    "model": "Full",
                    "seed": seed,
                    "attack": item["attack"],
                    "support": item["count"],
                    "recall": item["recall"],
                })

        del model, val, test
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return (
        pd.DataFrame(rows_gate),
        pd.DataFrame(rows_attention),
        pd.DataFrame(rows_beta),
        pd.DataFrame(rows_hard),
    )

def locate_subset_group_file(dataset_name: str, subset_fingerprint: str, filename: str) -> Optional[Path]:
    candidates = [
        STEP5_SUBSET_CACHE_ROOT / dataset_name / subset_fingerprint / filename,
        STEP5_SUBSET_CACHE_ROOT / dataset_name / filename,
    ]
    found = find_first_existing(candidates)
    if found:
        return found
    matches = list((STEP5_SUBSET_CACHE_ROOT / dataset_name).rglob(filename))
    return matches[0] if matches else None

def locate_subset_group_inverse(dataset_name: str, subset_fingerprint: str) -> Optional[Path]:
    return locate_subset_group_file(
        dataset_name, subset_fingerprint, "feature_group_inverse_int32.npy"
    )

def load_validation_threshold_for_prediction(prediction_path: Path) -> Optional[float]:
    metrics_path = prediction_path.parent / "test_metrics.json"
    if metrics_path.exists():
        result = load_json(metrics_path)
        validation = result.get("validation", {})
        if validation.get("threshold") is not None:
            return float(validation["threshold"])
    try:
        data = np.load(prediction_path, allow_pickle=True)
        if "threshold" in data.files:
            return float(np.ravel(data["threshold"])[0])
    except Exception:
        pass
    return None

def bootstrap_metric_ci(scores, labels, group_ids, threshold, replicates=1000, seed=2027):
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels, dtype=np.int64)
    group_ids = np.asarray(group_ids)
    unique_groups, inverse = np.unique(group_ids, return_inverse=True)

    # Build group members in O(n log n), avoiding one full scan per group.
    order = np.argsort(inverse, kind="stable")
    sorted_inverse = inverse[order]
    boundaries = np.flatnonzero(np.diff(sorted_inverse)) + 1
    members = np.split(order, boundaries)

    if threshold is None:
        base_for_threshold = evaluate_scores({"scores": scores, "labels": labels})
        threshold = float(base_for_threshold["threshold"])
        threshold_source = "test_fallback"
    else:
        threshold = float(threshold)
        threshold_source = "validation_saved"

    rng = np.random.default_rng(seed)
    values = []
    base = evaluate_scores(
        {"scores": scores, "labels": labels},
        threshold=threshold,
        score_sign=1,
    )
    for _ in range(replicates):
        sampled = rng.integers(0, len(members), size=len(members))
        idx = np.concatenate([members[i] for i in sampled])
        m = evaluate_scores(
            {"scores": scores[idx], "labels": labels[idx]},
            threshold=threshold,
            score_sign=1,
        )
        values.append((m["f1_macro"], m["pr_auc"], m["fp_rate"], m["tp_rate"]))
    arr = np.asarray(values, dtype=np.float64)
    output = {
        "threshold": threshold,
        "threshold_source": threshold_source,
    }
    estimates = [base["f1_macro"], base["pr_auc"], base["fp_rate"], base["tp_rate"]]
    for j, name in enumerate(["f1_macro", "pr_auc", "fp_rate", "tp_rate"]):
        output[name + "_estimate"] = float(estimates[j])
        output[name + "_ci_low"] = float(np.quantile(arr[:, j], 0.025))
        output[name + "_ci_high"] = float(np.quantile(arr[:, j], 0.975))
    return output


def locate_step4_summary(filename: str) -> Optional[Path]:
    candidates = [
        STEP4_ROOT / "paper" / filename,
        STEP4_ROOT / "confirmation" / filename,
        STEP4_ROOT / filename,
    ]
    found = find_first_existing(candidates)
    if found:
        return found
    matches = list(STEP4_ROOT.rglob(filename)) if STEP4_ROOT.exists() else []
    return sorted(matches, key=lambda p: len(str(p)))[0] if matches else None

def run_audit_phase():
    audit_dir = OUTPUT_ROOT / "audit"
    audit_dir.mkdir(parents=True, exist_ok=True)
    conflict_rows = []
    op_frames, gate_frames, attention_frames, beta_frames, hard_frames, bootstrap_frames = [], [], [], [], [], []

    audit_config = {
        "model_seeds": MODEL_SEEDS,
        "beta_values": BETA_VALUES,
        "bootstrap_replicates": BOOTSTRAP_REPLICATES,
        "bootstrap_seed": BOOTSTRAP_SEED,
        "hash_verify_mode": HASH_VERIFY_MODE,
        "hash_verify_sample_groups": HASH_VERIFY_SAMPLE_GROUPS,
    }
    audit_fp = config_fingerprint(audit_config)

    for dataset_name in DATASETS_TO_RUN:
        dataset_audit_dir = audit_dir / dataset_name
        dataset_audit_dir.mkdir(parents=True, exist_ok=True)
        marker = dataset_audit_dir / "completed.json"
        required = [
            dataset_audit_dir / "duplicate_conflict_audit.csv",
            dataset_audit_dir / "operating_points_all_seeds.csv",
            dataset_audit_dir / "gate_by_seed.csv",
            dataset_audit_dir / "semantic_attention_by_seed.csv",
            dataset_audit_dir / "beta_sensitivity_all_runs.csv",
            dataset_audit_dir / "hard_class_all_runs.csv",
            dataset_audit_dir / "hash_protocol_manifest.json",
        ]

        if marker.exists():
            completed = load_json(marker)
            if completed.get("config_fingerprint") == audit_fp and all(p.exists() for p in required):
                print("SKIP completed audit:", dataset_name)
                conflict_rows.extend(pd.read_csv(required[0]).to_dict(orient="records"))
                op_frames.append(pd.read_csv(required[1]))
                gate_frames.append(pd.read_csv(required[2]))
                attention_frames.append(pd.read_csv(required[3]))
                beta_frames.append(pd.read_csv(required[4]))
                hard_frames.append(pd.read_csv(required[5]))
                bootstrap_path = dataset_audit_dir / "group_bootstrap_ci.csv"
                if bootstrap_path.exists():
                    bootstrap_frames.append(pd.read_csv(bootstrap_path))
                update_master_progress("audit", dataset_name, "completed", {"reused": True})
                continue
            if completed.get("config_fingerprint") != audit_fp:
                raise RuntimeError(
                    f"Audit fingerprint mismatch for {dataset_name}. "
                    "Change STEP8_EXPERIMENT_TAG or remove the old audit directory."
                )

        print("\n" + "="*100)
        print("AUDIT:", dataset_name)
        print("="*100)
        preflight_step5(dataset_name)

        conflict = run_duplicate_conflict_audit(dataset_name)
        conflict_df_one = pd.DataFrame([conflict])
        conflict_df_one.to_csv(required[0], index=False)
        conflict_rows.append(conflict)

        op_one = aggregate_operating_points(dataset_name)
        op_one.to_csv(required[1], index=False)
        op_frames.append(op_one)

        bundle = load_fixed_subset_bundle(dataset_name)
        train_idx, val_idx, test_idx, scaler, split_meta = load_frozen_step5_artifacts(bundle)

        manifest = hash_protocol_manifest(dataset_name)
        inverse_path_for_verify = locate_subset_group_inverse(
            dataset_name, bundle.subset_fingerprint
        )
        counts_path_for_verify = locate_subset_group_file(
            dataset_name, bundle.subset_fingerprint, "feature_group_counts_int32.npy"
        )
        if inverse_path_for_verify and counts_path_for_verify:
            verify_result = verify_hash_groups_sample(
                bundle,
                np.asarray(np.load(inverse_path_for_verify, mmap_mode="r")),
                np.asarray(np.load(counts_path_for_verify, mmap_mode="r")),
                HASH_VERIFY_SAMPLE_GROUPS,
                seed=BOOTSTRAP_SEED,
            )
            verify_result["scope"] = "frozen Step-5 subset"
        else:
            verify_result = {
                "mode": HASH_VERIFY_MODE,
                "scope": "unavailable",
                "groups_checked": 0,
                "non_equal_hash_groups": None,
            }
        manifest["collision_verification_result"] = verify_result
        atomic_json_dump(manifest, required[6])

        gate, attention, beta, hard = run_checkpoint_diagnostics(
            dataset_name, bundle, train_idx, val_idx, test_idx, scaler
        )
        gate.to_csv(required[2], index=False)
        attention.to_csv(required[3], index=False)
        beta.to_csv(required[4], index=False)
        hard.to_csv(required[5], index=False)
        gate_frames.append(gate)
        attention_frames.append(attention)
        beta_frames.append(beta)
        hard_frames.append(hard)

        bootstrap_rows = []
        if RUN_GROUP_BOOTSTRAP:
            inverse_path = locate_subset_group_inverse(dataset_name, bundle.subset_fingerprint)
            if inverse_path is not None:
                subset_groups = np.asarray(np.load(inverse_path, mmap_mode="r"))
                test_group_ids = subset_groups[test_idx]
                for seed in MODEL_SEEDS:
                    for model_name, path in prediction_paths_for_table_xi(dataset_name, seed).items():
                        if path is None or not path.exists():
                            continue
                        data = np.load(path, allow_pickle=True)
                        if len(data["labels"]) != len(test_group_ids):
                            print("Skip bootstrap length mismatch:", model_name, seed)
                            continue
                        threshold = load_validation_threshold_for_prediction(path)
                        ci = bootstrap_metric_ci(
                            data["scores"], data["labels"], test_group_ids,
                            threshold,
                            BOOTSTRAP_REPLICATES, BOOTSTRAP_SEED + seed,
                        )
                        bootstrap_rows.append({
                            "dataset": dataset_name,
                            "model": model_name,
                            "seed": seed,
                            "n_groups": int(len(np.unique(test_group_ids))),
                            "replicates": BOOTSTRAP_REPLICATES,
                            **ci,
                        })
            else:
                print("Subset group inverse not found; bootstrap skipped:", dataset_name)
        bootstrap_one = pd.DataFrame(bootstrap_rows)
        bootstrap_one.to_csv(dataset_audit_dir / "group_bootstrap_ci.csv", index=False)
        bootstrap_frames.append(bootstrap_one)

        atomic_json_dump({
            "completed": True,
            "dataset": dataset_name,
            "config_fingerprint": audit_fp,
            "completed_at_unix": time.time(),
        }, marker)
        update_master_progress("audit", dataset_name, "completed")
        del bundle
        gc.collect()

    conflict_df = pd.DataFrame(conflict_rows)
    conflict_df.to_csv(audit_dir / "step8_duplicate_conflict_audit.csv", index=False)

    op_all = pd.concat(op_frames, ignore_index=True) if op_frames else pd.DataFrame()
    op_all.to_csv(audit_dir / "step8_operating_points_all_seeds.csv", index=False)
    table_xi = summarize_operating_points(op_all)
    table_xi.to_csv(audit_dir / "step8_table_XI_corrected.csv", index=False)
    if not table_xi.empty:
        (audit_dir / "step8_table_XI_corrected.tex").write_text(
            table_xi.to_latex(index=False, escape=False), encoding="utf-8"
        )

    gate_df = pd.concat(gate_frames, ignore_index=True) if gate_frames else pd.DataFrame()
    gate_df.to_csv(audit_dir / "step8_gate_by_seed.csv", index=False)
    if not gate_df.empty:
        gate_df.groupby("dataset", as_index=False).agg(
            gate_mean=("gate", "mean"),
            gate_std=("gate", lambda x: float(np.std(x, ddof=1)) if len(x)>1 else 0.0),
        ).to_csv(audit_dir / "step8_gate_summary.csv", index=False)

    attention_df = pd.concat(attention_frames, ignore_index=True) if attention_frames else pd.DataFrame()
    attention_df.to_csv(audit_dir / "step8_semantic_attention_by_seed.csv", index=False)

    beta_df = pd.concat(beta_frames, ignore_index=True) if beta_frames else pd.DataFrame()
    beta_df.to_csv(audit_dir / "step8_beta_sensitivity_all_runs.csv", index=False)
    if not beta_df.empty:
        beta_df.groupby(["dataset", "beta_classifier_weight"], as_index=False).agg(
            n_seeds=("seed", "nunique"),
            f1_mean=("test_f1_macro", "mean"),
            f1_std=("test_f1_macro", lambda x: float(np.std(x, ddof=1)) if len(x)>1 else 0.0),
            pr_auc_mean=("test_pr_auc", "mean"),
            pr_auc_std=("test_pr_auc", lambda x: float(np.std(x, ddof=1)) if len(x)>1 else 0.0),
            fp_rate_mean=("test_fp_rate", "mean"),
            fp_rate_std=("test_fp_rate", lambda x: float(np.std(x, ddof=1)) if len(x)>1 else 0.0),
            tp_rate_mean=("test_tp_rate", "mean"),
            tp_rate_std=("test_tp_rate", lambda x: float(np.std(x, ddof=1)) if len(x)>1 else 0.0),
        ).to_csv(audit_dir / "step8_beta_sensitivity_summary.csv", index=False)

    hard_df = pd.concat(hard_frames, ignore_index=True) if hard_frames else pd.DataFrame()
    hard_df.to_csv(audit_dir / "step8_hard_class_all_runs.csv", index=False)
    if not hard_df.empty:
        hard_df.groupby(["dataset", "model", "attack"], as_index=False).agg(
            n_seeds=("seed", "nunique"),
            support_mean=("support", "mean"),
            recall_mean=("recall", "mean"),
            recall_std=("recall", lambda x: float(np.std(x, ddof=1)) if len(x)>1 else 0.0),
        ).to_csv(audit_dir / "step8_hard_class_mean_std.csv", index=False)

    bootstrap_df = pd.concat(bootstrap_frames, ignore_index=True) if bootstrap_frames else pd.DataFrame()
    bootstrap_df.to_csv(audit_dir / "step8_group_bootstrap_ci.csv", index=False)

    # Copy existing Step-4 full-data summaries with mean ± std for all variants.
    per_attack_source = locate_step4_summary("exact_group_per_attack_summary.csv")
    model_summary_source = locate_step4_summary("exact_group_model_summary.csv")
    if per_attack_source:
        shutil.copy2(per_attack_source, audit_dir / "step8_full_data_per_attack_mean_std.csv")
    if model_summary_source:
        shutil.copy2(model_summary_source, audit_dir / "step8_full_data_model_mean_std.csv")

    update_master_progress("audit", status="completed")
    print("Audit outputs:", audit_dir)


In [9]:

# Cell 9 — Batch-partition sensitivity

def step5_full_result_as_partition_zero(dataset_name: str, seed: int) -> Dict[str, Any]:
    result = load_json(step5_full_task_dir(dataset_name, seed) / "test_metrics.json")
    return {
        "dataset": dataset_name,
        "experiment": "batch_partition_sensitivity",
        "partition_seed": 0,
        "model_seed": seed,
        "variant": "full",
        "source": "reused_step5_confirmation",
        "parameters": result.get("parameters"),
        "best_epoch": result.get("best_epoch"),
        "gate_final": result.get("gate_final"),
        "validation": result.get("validation", {}),
        "test": result.get("test", {}),
        "per_attack": result.get("per_attack", []),
    }

def run_batch_partition_phase():
    for dataset_name in DATASETS_TO_RUN:
        print("\n" + "="*100)
        print("BATCH-PARTITION SENSITIVITY:", dataset_name)
        print("="*100)
        preflight_step5(dataset_name)
        bundle = load_fixed_subset_bundle(dataset_name)
        train_idx, val_idx, test_idx, scaler, split_meta = load_frozen_step5_artifacts(bundle)
        y_train = bundle.labels[train_idx]
        positive_class_weight = min(
            int((y_train == 0).sum()) / max(int((y_train == 1).sum()), 1),
            POS_WEIGHT_CAP,
        )

        for partition_seed in BATCH_PARTITION_SEEDS:
            if partition_seed == 0:
                for model_seed in MODEL_SEEDS:
                    reused = step5_full_result_as_partition_zero(dataset_name, model_seed)
                    out_dir = batch_task_dir(dataset_name, 0, model_seed)
                    out_dir.mkdir(parents=True, exist_ok=True)
                    atomic_json_dump(reused, out_dir / "test_metrics.json")
                    atomic_json_dump({
                        "completed": True,
                        "reused": True,
                        "source": str(step5_full_task_dir(dataset_name, model_seed)),
                    }, out_dir / "completed.json")
                continue

            partition_dir = (
                OUTPUT_ROOT / "batch_partition_sensitivity" / dataset_name
                / f"partition_{partition_seed}" / "shared"
            )
            partition_dir.mkdir(parents=True, exist_ok=True)
            indices_path = partition_dir / "partition_indices.npz"
            if indices_path.exists():
                saved = np.load(indices_path)
                p_train = np.asarray(saved["train_idx"], dtype=np.int64)
                p_val = np.asarray(saved["val_idx"], dtype=np.int64)
                p_test = np.asarray(saved["test_idx"], dtype=np.int64)
            else:
                p_train = permute_split_for_partition(train_idx, partition_seed, 101)
                p_val = permute_split_for_partition(val_idx, partition_seed, 103)
                p_test = permute_split_for_partition(test_idx, partition_seed, 107)
                np.savez_compressed(
                    indices_path, train_idx=p_train, val_idx=p_val, test_idx=p_test
                )

            train_windows = prepare_graph_windows(bundle, p_train, scaler, f"Train p={partition_seed}")
            val_windows = prepare_graph_windows(bundle, p_val, scaler, f"Validation p={partition_seed}")
            test_windows = prepare_graph_windows(bundle, p_test, scaler, f"Test p={partition_seed}")

            for model_seed in MODEL_SEEDS:
                run_training_task(
                    bundle=bundle,
                    variant=FULL_VARIANT,
                    model_seed=model_seed,
                    train_windows=train_windows,
                    val_windows=val_windows,
                    test_windows=test_windows,
                    positive_class_weight=positive_class_weight,
                    task_dir=batch_task_dir(dataset_name, partition_seed, model_seed),
                    experiment_payload={
                        "experiment": "batch_partition_sensitivity",
                        "partition_seed": partition_seed,
                        "split_fingerprint": split_meta["split_fingerprint"],
                    },
                )
            del train_windows, val_windows, test_windows
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        update_master_progress("batch_partition", dataset_name, "completed")
        del bundle
        gc.collect()
    update_master_progress("batch_partition", status="completed")


In [10]:

# Cell 10 — Relation ablation

def run_relation_ablation_phase():
    for dataset_name in DATASETS_TO_RUN:
        print("\n" + "="*100)
        print("RELATION ABLATION:", dataset_name)
        print("="*100)
        preflight_step5(dataset_name)
        bundle = load_fixed_subset_bundle(dataset_name)
        train_idx, val_idx, test_idx, scaler, split_meta = load_frozen_step5_artifacts(bundle)
        y_train = bundle.labels[train_idx]
        positive_class_weight = min(
            int((y_train == 0).sum()) / max(int((y_train == 1).sum()), 1),
            POS_WEIGHT_CAP,
        )

        train_windows = prepare_graph_windows(bundle, train_idx, scaler, "Relation train")
        val_windows = prepare_graph_windows(bundle, val_idx, scaler, "Relation validation")
        test_windows = prepare_graph_windows(bundle, test_idx, scaler, "Relation test")

        for variant_name in RELATION_VARIANTS_TO_RUN:
            variant = RELATION_VARIANTS[variant_name]
            for model_seed in MODEL_SEEDS:
                run_training_task(
                    bundle=bundle,
                    variant=variant,
                    model_seed=model_seed,
                    train_windows=train_windows,
                    val_windows=val_windows,
                    test_windows=test_windows,
                    positive_class_weight=positive_class_weight,
                    task_dir=relation_task_dir(dataset_name, variant_name, model_seed),
                    experiment_payload={
                        "experiment": "relation_ablation",
                        "relation_variant": variant_name,
                        "partition_seed": 0,
                        "split_fingerprint": split_meta["split_fingerprint"],
                    },
                )

        # Reuse existing Full as "both_relations".
        for model_seed in MODEL_SEEDS:
            source = step5_full_task_dir(dataset_name, model_seed)
            target = relation_task_dir(dataset_name, "both_relations", model_seed)
            target.mkdir(parents=True, exist_ok=True)
            result = load_json(source / "test_metrics.json")
            atomic_json_dump({
                "dataset": dataset_name,
                "experiment": "relation_ablation",
                "relation_variant": "both_relations",
                "model_seed": model_seed,
                "variant": "full",
                "source": "reused_step5_confirmation",
                "parameters": result.get("parameters"),
                "best_epoch": result.get("best_epoch"),
                "gate_final": result.get("gate_final"),
                "validation": result.get("validation", {}),
                "test": result.get("test", {}),
                "per_attack": result.get("per_attack", []),
            }, target / "test_metrics.json")
            atomic_json_dump({
                "completed": True,
                "reused": True,
                "source": str(source),
            }, target / "completed.json")

        update_master_progress("relation_ablation", dataset_name, "completed")
        del train_windows, val_windows, test_windows, bundle
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    update_master_progress("relation_ablation", status="completed")


In [11]:

# Cell 11 — Aggregate all Step-8 results and manuscript-ready outputs

def normalize_result_file(path: Path, experiment: str) -> Optional[Dict[str, Any]]:
    if not path.exists():
        return None
    r = load_json(path)
    test = r.get("test", {})
    val = r.get("validation", {})
    return {
        "dataset": r.get("dataset"),
        "experiment": experiment,
        "partition_seed": r.get("partition_seed", 0),
        "relation_variant": r.get("relation_variant", r.get("variant")),
        "model_seed": r.get("model_seed", r.get("seed")),
        "variant": r.get("variant"),
        "source": r.get("source", "step8"),
        "parameters": r.get("parameters"),
        "best_epoch": r.get("best_epoch"),
        "gate_final": r.get("gate_final"),
        "threshold": val.get("threshold"),
        "f1_macro": test.get("f1_macro"),
        "auc_roc": test.get("auc_roc"),
        "pr_auc": test.get("pr_auc"),
        "fp_rate": test.get("fp_rate"),
        "tp_rate": test.get("tp_rate"),
        "tn": test.get("tn"),
        "fp": test.get("fp"),
        "fn": test.get("fn"),
        "tp": test.get("tp"),
        "path": str(path),
    }

def aggregate_step8():
    rows, attack_rows, completeness = [], [], []

    for dataset in DATASETS_TO_RUN:
        for partition_seed in BATCH_PARTITION_SEEDS:
            for seed in MODEL_SEEDS:
                path = batch_task_dir(dataset, partition_seed, seed) / "test_metrics.json"
                completed = batch_task_dir(dataset, partition_seed, seed) / "completed.json"
                completeness.append({
                    "dataset": dataset,
                    "experiment": "batch_partition_sensitivity",
                    "configuration": f"partition_{partition_seed}",
                    "seed": seed,
                    "complete": path.exists() and completed.exists(),
                    "path": str(path),
                })
                row = normalize_result_file(path, "batch_partition_sensitivity")
                if row:
                    rows.append(row)
                    for item in load_json(path).get("per_attack", []):
                        attack_rows.append({
                            "dataset": dataset,
                            "experiment": "batch_partition_sensitivity",
                            "configuration": f"partition_{partition_seed}",
                            "seed": seed,
                            "attack": item["attack"],
                            "support": item["count"],
                            "recall": item["recall"],
                        })

        for variant in ["both_relations"] + RELATION_VARIANTS_TO_RUN:
            for seed in MODEL_SEEDS:
                path = relation_task_dir(dataset, variant, seed) / "test_metrics.json"
                completed = relation_task_dir(dataset, variant, seed) / "completed.json"
                completeness.append({
                    "dataset": dataset,
                    "experiment": "relation_ablation",
                    "configuration": variant,
                    "seed": seed,
                    "complete": path.exists() and completed.exists(),
                    "path": str(path),
                })
                row = normalize_result_file(path, "relation_ablation")
                if row:
                    rows.append(row)
                    for item in load_json(path).get("per_attack", []):
                        attack_rows.append({
                            "dataset": dataset,
                            "experiment": "relation_ablation",
                            "configuration": variant,
                            "seed": seed,
                            "attack": item["attack"],
                            "support": item["count"],
                            "recall": item["recall"],
                        })

    complete_df = pd.DataFrame(completeness)
    complete_df.to_csv(OUTPUT_ROOT / "step8_run_completeness.csv", index=False)
    print("Completeness:", int(complete_df["complete"].sum()), "/", len(complete_df))
    if not complete_df["complete"].all():
        display(complete_df.loc[~complete_df["complete"]])

    runs = pd.DataFrame(rows)
    runs.to_csv(OUTPUT_ROOT / "step8_all_new_experiment_runs.csv", index=False)
    if not runs.empty:
        summary_keys = ["dataset", "experiment", "partition_seed", "relation_variant"]
        summary = runs.groupby(summary_keys, dropna=False, as_index=False).agg(
            n_seeds=("model_seed", "nunique"),
            parameters=("parameters", "mean"),
            best_epoch_mean=("best_epoch", "mean"),
            f1_macro_mean=("f1_macro", "mean"),
            f1_macro_std=("f1_macro", lambda x: float(np.std(x, ddof=1)) if len(x)>1 else 0.0),
            auc_roc_mean=("auc_roc", "mean"),
            auc_roc_std=("auc_roc", lambda x: float(np.std(x, ddof=1)) if len(x)>1 else 0.0),
            pr_auc_mean=("pr_auc", "mean"),
            pr_auc_std=("pr_auc", lambda x: float(np.std(x, ddof=1)) if len(x)>1 else 0.0),
            fp_rate_mean=("fp_rate", "mean"),
            fp_rate_std=("fp_rate", lambda x: float(np.std(x, ddof=1)) if len(x)>1 else 0.0),
            tp_rate_mean=("tp_rate", "mean"),
            tp_rate_std=("tp_rate", lambda x: float(np.std(x, ddof=1)) if len(x)>1 else 0.0),
            gate_mean=("gate_final", "mean"),
            gate_std=("gate_final", lambda x: float(np.std(x, ddof=1)) if len(x)>1 else 0.0),
        )
        summary.to_csv(OUTPUT_ROOT / "step8_experiment_summary.csv", index=False)
        display(summary)

    attacks = pd.DataFrame(attack_rows)
    attacks.to_csv(OUTPUT_ROOT / "step8_per_attack_all_runs.csv", index=False)
    if not attacks.empty:
        attacks.groupby(
            ["dataset", "experiment", "configuration", "attack"], as_index=False
        ).agg(
            n_seeds=("seed", "nunique"),
            support_mean=("support", "mean"),
            recall_mean=("recall", "mean"),
            recall_std=("recall", lambda x: float(np.std(x, ddof=1)) if len(x)>1 else 0.0),
        ).to_csv(OUTPUT_ROOT / "step8_per_attack_mean_std.csv", index=False)

    # Manuscript fill guide
    lines = [
        "# Step 8 manuscript fill guide",
        "",
        "## Required wording",
        "- Call the 70/30 subset an **attack-enriched frozen benchmark**.",
        "- Do not interpret prevalence-sensitive precision/PR-AUC as deployment-prevalence estimates.",
        "- Call Full-LOO a **cross-fitted no-self sensitivity diagnostic**, not a causal leave-one-out proof.",
        "- Use **identical 39-dimensional local numerical feature vectors**, not identical full model inputs.",
        "- Call the communication relation **bidirected with shared reverse-direction parameters**.",
        "- Call the gate a **learned global scalar gate**.",
        "",
        "## Generated files",
        "- `audit/step8_table_XI_corrected.csv`",
        "- `audit/step8_duplicate_conflict_audit.csv`",
        "- `audit/step8_gate_summary.csv`",
        "- `audit/step8_semantic_attention_by_seed.csv`",
        "- `audit/step8_beta_sensitivity_summary.csv`",
        "- `audit/step8_hard_class_mean_std.csv`",
        "- `step8_experiment_summary.csv`",
        "- `step8_per_attack_mean_std.csv`",
        "",
    ]
    (OUTPUT_ROOT / "step8_manuscript_fill_guide.md").write_text(
        "\n".join(lines), encoding="utf-8"
    )

    # Compact bundle excludes checkpoints/predictions.
    bundle_path = OUTPUT_ROOT / "THAT_IDS_step8_results_bundle.zip"
    allowed = {".csv", ".json", ".md", ".tex"}
    with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as z:
        for path in OUTPUT_ROOT.rglob("*"):
            if path.is_file() and path != bundle_path and path.suffix.lower() in allowed:
                z.write(path, arcname=str(path.relative_to(OUTPUT_ROOT)))
    print("Bundle:", bundle_path)
    update_master_progress("aggregate", status="completed", details={
        "complete_runs": int(complete_df["complete"].sum()),
        "expected_runs": int(len(complete_df)),
    })


In [12]:

# Cell 12 — RUN ALL orchestrator

def run_selected_phases():
    started = time.time()
    update_master_progress("master", status="running", details={"run_mode": RUN_MODE})

    if RUN_MODE in {"audit", "all"}:
        run_audit_phase()

    if RUN_MODE in {"batch_partition", "all"}:
        run_batch_partition_phase()

    if RUN_MODE in {"relation_ablation", "all"}:
        run_relation_ablation_phase()

    if RUN_MODE in {"aggregate", "all"}:
        aggregate_step8()

    update_master_progress("master", status="completed", details={
        "run_mode": RUN_MODE,
        "elapsed_hours": (time.time() - started) / 3600.0,
    })
    print(f"\nStep 8 completed for RUN_MODE={RUN_MODE}.")
    print(f"Elapsed: {(time.time()-started)/3600:.2f} hours")
    print("Output:", OUTPUT_ROOT)

run_selected_phases()



AUDIT: CIC_V2
Preflight OK: CIC_V2

Loading frozen subset CIC_V2: 496,384 rows
  chunks=10 | selected=131,692/496,384
  chunks=20 | selected=263,031/496,384
  chunks=30 | selected=394,663/496,384
  chunks=38 | selected=496,384/496,384
Loaded in 1.6 min | shape=(496384, 39)
Frozen split: train=361,202, val=40,519, test=94,663 | 58cfcf6084a427575c09
    Diagnostic validation: 9/9
  Diagnostic validation: 9 batches in 0.0 min
    Diagnostic test: 19/19
  Diagnostic test: 19 batches in 0.0 min

AUDIT: UNSW_V2
Preflight OK: UNSW_V2

Loading frozen subset UNSW_V2: 245,994 rows
  chunks=5 | selected=245,994/245,994
Loaded in 0.2 min | shape=(245994, 39)
Frozen split: train=174,756, val=23,297, test=47,941 | fdfa40e95ad722f3315a
    Diagnostic validation: 5/5
  Diagnostic validation: 5 batches in 0.0 min
    Diagnostic test: 10/10
  Diagnostic test: 10 batches in 0.0 min
Audit outputs: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step8_supervisor_revision_master_v1/audit

BATCH-PARTITION 

,dataset,experiment,partition_seed,relation_variant,n_seeds,parameters,best_epoch_mean,f1_macro_mean,f1_macro_std,auc_roc_mean,auc_roc_std,pr_auc_mean,pr_auc_std,fp_rate_mean,fp_rate_std,tp_rate_mean,tp_rate_std,gate_mean,gate_std
0,CIC_V2,batch_partition_sensitivity,0,full,3,157196.0,21.000000,0.973577,0.001140,0.990513,0.001349,0.983966,0.001665,0.000223,0.000056,0.922586,0.003318,0.185110,0.000142
1,CIC_V2,batch_partition_sensitivity,3027,full,3,157196.0,20.333333,0.972529,0.001552,0.990986,0.000783,0.984502,0.001014,0.000260,0.000063,0.919689,0.004579,0.185287,0.000074
2,CIC_V2,batch_partition_sensitivity,4027,full,3,157196.0,22.333333,0.974674,0.000700,0.990513,0.001845,0.983843,0.002456,0.000278,0.000050,0.925908,0.001867,0.185082,0.000135
3,CIC_V2,relation_ablation,0,both_no_protocol_numeric,3,155850.0,15.000000,0.972808,0.001122,0.989652,0.000398,0.982898,0.000485,0.000306,0.000037,0.920625,0.003262,0.185186,0.000257
4,CIC_V2,relation_ablation,0,both_relations,3,157196.0,21.000000,0.973577,0.001140,0.990513,0.001349,0.983966,0.001665,0.000223,0.000056,0.922586,0.003318,0.185110,0.000142
5,CIC_V2,relation_ablation,0,ip_ip_only,3,156811.0,64.666667,0.977446,0.000172,0.993946,0.000070,0.988856,0.000168,0.001210,0.000639,0.936723,0.002415,0.181668,0.000245
6,CIC_V2,relation_ablation,0,ip_protocol_only,3,156811.0,79.333333,0.945228,0.032311,0.991391,0.000163,0.985467,0.000172,0.000102,0.000016,0.845187,0.085737,0.182360,0.000070
7,UNSW_V2,batch_partition_sensitivity,0,full,3,157196.0,57.333333,0.994810,0.001947,0.999194,0.000054,0.997421,0.000213,0.006057,0.002207,0.999145,0.000656,0.184268,0.000127
8,UNSW_V2,batch_partition_sensitivity,3027,full,3,157196.0,67.333333,0.994773,0.002431,0.999262,0.000040,0.997777,0.000173,0.006420,0.003067,0.999842,0.000078,0.184139,0.000153
9,UNSW_V2,batch_partition_sensitivity,4027,full,3,157196.0,63.333333,0.994651,0.002494,0.999258,0.000012,0.997728,0.000053,0.006540,0.003173,0.999775,0.000217,0.184189,0.000108


Bundle: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step8_supervisor_revision_master_v1/THAT_IDS_step8_results_bundle.zip

Step 8 completed for RUN_MODE=all.
Elapsed: 0.72 hours
Output: /content/drive/MyDrive/THAT_IDS_RERUN/conference_step8_supervisor_revision_master_v1
